# 📱 TP3 — Prédire le Départ d'un Client Télécom (Churn — Classification)
## Module Data Science — Machine Learning | Corrigé détaillé
**Dataset** : `clients_telecom.csv` (8 000 clients) — cible : `churn` (0/1, ~25%)

In [ ]:
!pip install scikit-learn seaborn -q
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)
sns.set_theme(style='whitegrid')

## 1. Exploration (EDA)

In [ ]:
df = pd.read_csv("clients_telecom.csv")
print("Dimensions :", df.shape)
df.head()

In [ ]:
df.info()
print("Manquants :\n", df.isna().sum())
print("\nTaux de churn :", round(df["churn"].mean()*100, 1), "%")
print(df["churn"].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18,5))
df.groupby("nb_reclamations")["churn"].mean().plot(kind="bar", ax=axes[0], color="orange")
axes[0].set_title("Churn selon nb réclamations"); axes[0].set_ylabel("Taux de churn")
sns.boxplot(data=df, x="churn", y="anciennete_mois", ax=axes[1])
axes[1].set_title("Ancienneté selon churn"); axes[1].set_xticklabels(["Reste","Part"])
sns.boxplot(data=df, x="churn", y="facture_mensuelle", ax=axes[2])
axes[2].set_title("Facture selon churn"); axes[2].set_xticklabels(["Reste","Part"])
plt.tight_layout(); plt.show()

> 🔑 Hypothèses : plus de réclamations + faible ancienneté + facture élevée = plus de churn.

## 2. Préparation (exclure client_id)

In [ ]:
features = ["age", "anciennete_mois", "conso_data_go", "minutes_appel",
            "nb_sms", "facture_mensuelle", "nb_reclamations"]   # client_id EXCLU
X = df[features]
y = df["churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f"Train : {X_train.shape[0]} | Test : {X_test.shape[0]}")

> ⚠️ `stratify=y` est essentiel : la classe « part » est minoritaire (~25%).

## 3. Les 3 modèles de classification

In [ ]:
resultats = {}
logreg = LogisticRegression(max_iter=500).fit(X_train_s, y_train)
resultats["Régression logistique"] = accuracy_score(y_test, logreg.predict(X_test_s))
arbre = DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_train, y_train)
resultats["Arbre de décision"] = accuracy_score(y_test, arbre.predict(X_test))
knn = KNeighborsClassifier(n_neighbors=7).fit(X_train_s, y_train)
resultats["KNN (k=7)"] = accuracy_score(y_test, knn.predict(X_test_s))
for m, s in sorted(resultats.items(), key=lambda x: x[1], reverse=True):
    print(f"{m:25} : {s:.3f}")

> ⚠️ ~0.86 semble bon, mais 75% des clients restent : la précision globale est **trompeuse**. Regarder le **recall** de la classe « part ».

## 4. Évaluation approfondie (le recall !)

In [ ]:
pred = logreg.predict(X_test_s)
cm = confusion_matrix(y_test, pred)
ConfusionMatrixDisplay(cm, display_labels=["Reste", "Part"]).plot(cmap="Oranges")
plt.title("Matrice de confusion — Churn"); plt.show()
print(classification_report(y_test, pred, target_names=["Reste", "Part"]))

> 🔑 Précision globale ~0.86 mais **recall de « Part » ~0.61** → ~40% des partants ratés ! En problème déséquilibré, c'est le recall de la classe minoritaire qui compte.

## 5. Réflexion métier (Q13)

- **Faux négatif** = ne pas détecter un partant → **client perdu**, très coûteux 💸
- **Faux positif** = fausse alerte sur un fidèle → coût d'une offre inutile, faible

> 🔑 On privilégie le **recall** de « Part » (détecter un max de partants). C'est l'inverse du TP2 crédit !

## 6. Interpréter et agir

In [ ]:
importances = pd.Series(arbre.feature_importances_, index=features).sort_values(ascending=False)
plt.figure(figsize=(9,5)); importances.plot(kind="barh")
plt.title("Facteurs de churn"); plt.gca().invert_yaxis(); plt.show()
print(importances.head(5))

In [ ]:
def predire_churn(modele, scaler, colonnes, **infos):
    client = pd.DataFrame(0, index=[0], columns=colonnes)
    for cle, val in infos.items():
        if cle in client.columns:
            client[cle] = val
    client_s = scaler.transform(client)
    proba = modele.predict_proba(client_s)[0][1]
    return ("À RISQUE" if proba > 0.5 else "FIDÈLE"), proba

statut, proba = predire_churn(
    logreg, scaler, features,
    age=30, anciennete_mois=6, conso_data_go=3, minutes_appel=100,
    nb_sms=20, facture_mensuelle=45000, nb_reclamations=5)
print(f"Client récent mécontent → {statut} (proba de départ : {proba:.1%})")

## 7. Actions de fidélisation + Conclusion

- 📞 Beaucoup de réclamations → renforcer le SAV
- 🎁 Nouveaux clients → programme d'accueil / onboarding
- 💡 Grosses factures → forfait mieux adapté

Meilleur modèle : **régression logistique** (~0.86), mais le vrai enjeu est le **recall de « Part » (~0.61)**.